In [3]:
%load_ext autoreload
%autoreload 2

from torch.utils.data import DataLoader
from shapely.geometry import Point
import matplotlib.pyplot as plt
import math
import numpy as np

from extract.extractors.georeferencers import RoadMatcherGeoreferencer


from dataset import SatMapDataset, graph_collate_fn
from utils import load_config

In [4]:
config_path = "config/toponet_vitb_256_os.yaml"
config = load_config(config_path)

In [5]:
val_ds = SatMapDataset(config, is_train=False,return_graph=False,return_metadata=True)

In [6]:
# val_loader = DataLoader(
#     val_ds,
#     batch_size=1,
#     shuffle=True,
#     num_workers=config.DATA_WORKER_NUM,
#     pin_memory=True,
# )

In [7]:
georeferencer = RoadMatcherGeoreferencer(sam_config_path=config_path,debug_mode=True)
print(georeferencer.mask_extractor.sam.device)

2025-03-18 15:43:18 - root - INFO - SAM Matched params: ['image_encoder.pos_embed', 'image_encoder.patch_embed.proj.weight', 'image_encoder.patch_embed.proj.bias', 'image_encoder.blocks.0.norm1.weight', 'image_encoder.blocks.0.norm1.bias', 'image_encoder.blocks.0.attn.rel_pos_h', 'image_encoder.blocks.0.attn.rel_pos_w', 'image_encoder.blocks.0.attn.qkv.weight', 'image_encoder.blocks.0.attn.qkv.bias', 'image_encoder.blocks.0.attn.proj.weight', 'image_encoder.blocks.0.attn.proj.bias', 'image_encoder.blocks.0.norm2.weight', 'image_encoder.blocks.0.norm2.bias', 'image_encoder.blocks.0.mlp.lin1.weight', 'image_encoder.blocks.0.mlp.lin1.bias', 'image_encoder.blocks.0.mlp.lin2.weight', 'image_encoder.blocks.0.mlp.lin2.bias', 'image_encoder.blocks.1.norm1.weight', 'image_encoder.blocks.1.norm1.bias', 'image_encoder.blocks.1.attn.rel_pos_h', 'image_encoder.blocks.1.attn.rel_pos_w', 'image_encoder.blocks.1.attn.qkv.weight', 'image_encoder.blocks.1.attn.qkv.bias', 'image_encoder.blocks.1.attn.pro

cuda:0


In [8]:

# Create lists to store data for analysis
original_points = []
offset_points = []
new_points = []
distances = []

# Process all data points
for i, datum in enumerate(val_ds):
    # Get the actual metadata for the current data point
    try:
        metadata = datum['metadata']
    
    
        # Extract original center point
        original_lat = metadata['center']['lat']
        original_lon = metadata['center']['lon']
        original_point = (original_lon, original_lat)
        
        # Extract offset point
        offset_lat = metadata['offset']['lat']
        offset_lon = metadata['offset']['lon']
        offset_point = Point(offset_lon, offset_lat)
        
        # Apply georeferencing to get corrected point
        new_point = georeferencer.georeference(datum['rgb'], offset_point,zoom=16)
        
        # Calculate distance between new point and original point (in meters)
        # Convert to Point for consistent format
        original_point_obj = Point(original_point)
        new_point_obj = Point(new_point)
        
        # Approximate distance calculation (Haversine formula would be more accurate)
        # 1 degree ≈ 111,000 meters
        distance_meters = (
            ((new_point[0] - original_point[0]) * 111000 * math.cos(math.radians(original_lat)))**2 + 
            ((new_point[1] - original_point[1]) * 111000)**2
        )**0.5
        
        # Store data for analysis
        original_points.append(original_point)
        offset_points.append((offset_lon, offset_lat))
        new_points.append(new_point)
        distances.append(distance_meters)
        
        # Print detailed information for first few samples
        
        print(f"\n--- Sample {i+1} ---")
        print(f"Original point: {original_point}")
        print(f"Offset point:   {offset_point.coords[0]}")
        print(f"New point:      {new_point}")
        print(f"Distance error: {distance_meters:.2f} meters")
        print(f"Offset amount:  {metadata['offset']['distance']:.2f} meters")
        if i>15:
            break
    except:
        continue

# Convert to numpy arrays for analysis
distances = np.array(distances)

# Print statistical analysis
print("\n=== Statistical Analysis ===")
print(f"Number of samples: {len(distances)}")
print(f"Mean error distance: {np.mean(distances):.2f} meters")
print(f"Median error distance: {np.median(distances):.2f} meters")
print(f"Min error distance: {np.min(distances):.2f} meters")
print(f"Max error distance: {np.max(distances):.2f} meters")
print(f"Standard deviation: {np.std(distances):.2f} meters")

# Success rate (within certain thresholds)
within_10m = np.sum(distances < 10) / len(distances) * 100
within_25m = np.sum(distances < 25) / len(distances) * 100
within_50m = np.sum(distances < 50) / len(distances) * 100

print(f"\n=== Accuracy ===")
print(f"Points within 10m: {within_10m:.1f}%")
print(f"Points within 25m: {within_25m:.1f}%")
print(f"Points within 50m: {within_50m:.1f}%")

# Create a histogram of distances
plt.figure(figsize=(10, 6))
plt.hist(distances, bins=20, alpha=0.7)
plt.axvline(np.median(distances), color='r', linestyle='dashed', linewidth=1, label=f'Median: {np.median(distances):.2f}m')
plt.axvline(np.mean(distances), color='g', linestyle='dashed', linewidth=1, label=f'Mean: {np.mean(distances):.2f}m')
plt.xlabel('Error Distance (meters)')
plt.ylabel('Frequency')
plt.title('Distribution of Georeferencing Errors')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Optionally, create a scatter plot comparing original vs corrected positions
plt.figure(figsize=(10, 10))
original_lons, original_lats = zip(*[(p[0], p[1]) for p in original_points])
new_lons, new_lats = zip(*[(p[0], p[1]) for p in new_points])
plt.scatter(original_lons, original_lats, alpha=0.5, label='Original Points')
plt.scatter(new_lons, new_lats, alpha=0.5, label='Corrected Points')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Original vs Corrected Positions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

Debug mode enabled. Saving outputs to: debug_outputs/26510


/home/alex.martin/sam_road_map/.venv/lib/python3.12/site-packages/extract/extractors/georeferencers/mask_generators.py:290: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  patches.append(torch.tensor(patch, dtype=torch.float32))



--- Sample 1 ---
Original point: (-1.17828369140625, 54.46205624742783)
Offset point:   (-1.1783796867016492, 54.462070042070934)
New point:      [-1.17511368 54.46247028]
Distance error: 209.62 meters
Offset amount:  6.39 meters
Debug mode enabled. Saving outputs to: debug_outputs/59531


/home/alex.martin/sam_road_map/.venv/lib/python3.12/site-packages/extract/extractors/georeferencers/mask_generators.py:290: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  patches.append(torch.tensor(patch, dtype=torch.float32))



--- Sample 2 ---
Original point: (-2.56256103515625, 54.32453296307182)
Offset point:   (-2.562682130649108, 54.3246142438544)
New point:      [-2.56252809 54.32452944]
Distance error: 2.17 meters
Offset amount:  11.96 meters
Debug mode enabled. Saving outputs to: debug_outputs/88174


...


--- Sample 15 ---
Original point: (-1.78253173828125, 53.62998192140979)
Offset point:   (-1.7822289033255327, 53.629997468986964)
New point:      [-1.78253447 53.62996178]
Distance error: 2.24 meters
Offset amount:  20.03 meters
Debug mode enabled. Saving outputs to: debug_outputs/56002


/home/alex.martin/sam_road_map/.venv/lib/python3.12/site-packages/extract/extractors/georeferencers/mask_generators.py:290: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  patches.append(torch.tensor(patch, dtype=torch.float32))



--- Sample 16 ---
Original point: (-1.48040771484375, 53.54520363618777)
Offset point:   (-1.4805051388651234, 53.54504803797065)
New point:      [-1.48039922 53.54517695]
Distance error: 3.02 meters
Offset amount:  18.45 meters
Debug mode enabled. Saving outputs to: debug_outputs/54559


/home/alex.martin/sam_road_map/.venv/lib/python3.12/site-packages/extract/extractors/georeferencers/mask_generators.py:290: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  patches.append(torch.tensor(patch, dtype=torch.float32))


Debug mode enabled. Saving outputs to: debug_outputs/48878

--- Sample 18 ---
Original point: (-1.17828369140625, 53.515817672234405)
Offset point:   (-1.1781784568199545, 53.51584422796334)
New point:      [-1.1782509  53.51579897]
Distance error: 3.00 meters
Offset amount:  7.55 meters

=== Statistical Analysis ===
Number of samples: 15
Mean error distance: 79.98 meters
Median error distance: 33.87 meters
Min error distance: 2.17 meters
Max error distance: 322.57 meters
Standard deviation: 106.66 meters

=== Accuracy ===
Points within 10m: 46.7%
Points within 25m: 46.7%
Points within 50m: 66.7%


In [9]:
import torch
print(torch.cuda.is_available())

True
